In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.3 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

Saving bigger_dataset.zip to bigger_dataset.zip


In [3]:
import zipfile

zip_path = "bigger_dataset.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("dataset")

In [4]:
import os
import random
import shutil

base_path = "/content/dataset/train"

images_path = os.path.join(base_path, "images")
labels_path = os.path.join(base_path, "labels")

# crea val dirs
val_img = "/content/dataset/valid/images"
val_lbl = "/content/dataset/valid/labels"

os.makedirs(val_img, exist_ok=True)
os.makedirs(val_lbl, exist_ok=True)

images = os.listdir(images_path)
random.shuffle(images)

split = int(len(images) * 0.2)  # 20% validation

val_images = images[:split]

for img in val_images:
    # image
    shutil.move(
        os.path.join(images_path, img),
        os.path.join(val_img, img)
    )

    # label
    label = img.replace(".jpg", ".txt").replace(".jpeg", ".txt").replace(".png", ".txt")

    if os.path.exists(os.path.join(labels_path, label)):
        shutil.move(
            os.path.join(labels_path, label),
            os.path.join(val_lbl, label)
        )

In [6]:

from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/dataset/data.yaml",
    epochs=150,           # da 20 a 150 — vuoi memorizzare quel cartello
    imgsz=640,
    batch=8,              # batch più piccolo = aggiornamenti più frequenti su pochi sample
    freeze=6,             # sblocca più layer (prima eri a 10) per adattarsi meglio
    device=0,

    # Learning rate
    lr0=0.001,            # più basso del default (0.01) per fine-tuning stabile
    lrf=0.01,             # lr finale = lr0 * lrf
    warmup_epochs=3,

    # Optimizer
    optimizer="AdamW",    # meglio di SGD per dataset piccoli
    weight_decay=0.0005,

    # Augmentation aggressiva (compensa dataset piccolo)
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,            # simula variazioni di luce
    degrees=10,           # rotazioni leggere
    translate=0.1,
    scale=0.5,            # zoom in/out
    fliplr=0.0,           # NON flippare — la freccia ha direzione
    flipud=0.0,           # NON flippare verticalmente
    mosaic=0.5,           # ridotto, può confondere su 1 sola classe
    mixup=0.0,            # disattiva, inutile con 1 classe
    copy_paste=0.3,       # incolla il cartello in background diversi

    # Salvataggio
    save_period=10,
    patience=30,          # early stopping se non migliora in 30 epoche
    val=True,
)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=6, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=30, perspe

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78b105fe94c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [7]:
from google.colab import files

files.download("/content/runs/detect/train-2/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>